# Tata Steel – Stock Market and Financial Statement Analysis

This notebook analyses Tata Steel historical stock data and financial statements.

**Input files expected in the same folder as this notebook:**
- `TATASTEEL_Historical_Data.csv`
- `TATASTEEL_Financial_Data.xlsx`

Run the cells from top to bottom. If your files have different names, update the file paths in the data-loading cell.


## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline
sns.set_theme(style="whitegrid")


## 2. Load the datasets

In [ ]:
stock_file = "TATASTEEL_Historical_Data.csv"
financial_file = "TATASTEEL_Financial_Data.xlsx"

stock_raw = pd.read_csv(stock_file)
financial_sheets = pd.read_excel(financial_file, sheet_name=None)

print("Stock data shape:", stock_raw.shape)
print("Available financial sheets:", list(financial_sheets.keys()))

display(stock_raw.head())


## 3. Clean and prepare historical stock data

In [ ]:
# Clean and prepare historical stock data.
stock = stock_raw.copy()
stock.columns = [str(c).strip() for c in stock.columns]

# In this CSV, the date field is named 'Price'. Rename it to 'Date'.
if "Price" in stock.columns and "Date" not in stock.columns:
    stock = stock.rename(columns={"Price": "Date"})

# Some downloaded stock CSV files contain metadata rows such as 'Ticker' and 'Date'.
# Keep only rows where the date field can be parsed as a real date.
if "Date" not in stock.columns:
    raise ValueError("No date column found. Expected 'Date' or 'Price' in the CSV.")

stock["Date"] = pd.to_datetime(stock["Date"], errors="coerce")
stock = stock.dropna(subset=["Date"]).copy()

# Convert stock-price and volume fields to numeric values.
for col in ["Open", "High", "Low", "Close", "Volume"]:
    if col in stock.columns:
        stock[col] = pd.to_numeric(
            stock[col].astype(str).str.replace(",", "", regex=False),
            errors="coerce"
        )

# Remove rows without a closing price, sort chronologically and remove duplicates.
if "Close" not in stock.columns:
    raise ValueError("The stock CSV must contain a 'Close' column.")

stock = stock.dropna(subset=["Close"])
stock = stock.sort_values("Date").drop_duplicates().reset_index(drop=True)

print("Cleaned stock data shape:", stock.shape)
print("Date range:", stock["Date"].min().date(), "to", stock["Date"].max().date())
display(stock.head())
display(stock.tail())
stock.info()

## 4. Explore stock data and descriptive statistics

In [ ]:
display(stock.describe())

print("Missing values by column:")
display(stock.isna().sum())

print("Duplicate rows:", stock.duplicated().sum())

## 5. Calculate daily returns and moving averages

In [ ]:
if "Close" in stock.columns:
    stock["Daily_Return"] = stock["Close"].pct_change() * 100
    stock["MA_20"] = stock["Close"].rolling(window=20).mean()
    stock["MA_50"] = stock["Close"].rolling(window=50).mean()

    display(stock[["Close", "Daily_Return", "MA_20", "MA_50"]].tail())

    daily_volatility = stock["Daily_Return"].std()
    annualized_volatility = daily_volatility * np.sqrt(252)
    print(f"Daily return standard deviation: {daily_volatility:.2f}%")
    print(f"Annualized historical volatility (approx.): {annualized_volatility:.2f}%")


## 6. Analyse financial statement sheets

In [ ]:
# Display a preview of each worksheet.
for sheet_name, frame in financial_sheets.items():
    print("\\n---", sheet_name, "---")
    print("Shape:", frame.shape)
    display(frame.head())


### Prepare financial statement data

The workbook may use different layouts for each statement. Inspect the previews above and adjust the column names below if required. The following cell performs basic cleaning without assuming a particular financial-year layout.


In [ ]:
financial_clean = {}

for sheet_name, frame in financial_sheets.items():
    temp = frame.copy()
    temp.columns = [str(c).strip() for c in temp.columns]
    temp = temp.dropna(how="all")
    financial_clean[sheet_name] = temp

    print("\\nNumeric summary:", sheet_name)
    numeric = temp.select_dtypes(include=np.number)
    if not numeric.empty:
        display(numeric.describe())
    else:
        print("No numeric columns detected automatically.")

### Revenue and net income trends (if available)\n\nThis section looks for common revenue and net-income row labels in the Income Statement. It plots the values only when matching rows and financial-year columns are available. Check the source workbook for units before interpreting the amounts.

In [ ]:
income = financial_clean.get("Income Statement")

if income is None:
    print("No sheet named 'Income Statement' was found.")
else:
    # The first column normally contains financial item names; remaining columns are years.
    item_col = income.columns[0]
    year_cols = []
    for col in income.columns[1:]:
        parsed_year = pd.to_datetime(str(col), errors="coerce")
        if pd.notna(parsed_year):
            year_cols.append(col)

    def find_financial_row(frame, possible_names):
        labels = frame[item_col].astype(str).str.strip().str.lower()
        for name in possible_names:
            matches = frame.loc[labels == name.lower()]
            if not matches.empty:
                return matches.iloc[0]
        # Also allow common label variants containing the search phrase.
        for name in possible_names:
            matches = frame.loc[labels.str.contains(name.lower(), regex=False, na=False)]
            if not matches.empty:
                return matches.iloc[0]
        return None

    revenue_row = find_financial_row(
        income, ["Total Revenue", "Operating Revenue", "Revenue"]
    )
    net_income_row = find_financial_row(
        income, ["Net Income", "Net Income Common Stockholders", "Net Profit"]
    )

    def plot_financial_series(row, chart_title, y_label):
        if row is None or not year_cols:
            print(chart_title + ": matching data was not found in the workbook.")
            return

        years = [pd.to_datetime(str(c)).year for c in year_cols]
        values = pd.to_numeric(row[year_cols], errors="coerce")
        valid = pd.DataFrame({"Year": years, "Value": values}).dropna()
        valid = valid.sort_values("Year")

        if valid.empty:
            print(chart_title + ": no numeric values were available.")
            return

        plt.figure(figsize=(9, 5))
        plt.plot(valid["Year"].astype(str), valid["Value"], marker="o")
        plt.xlabel("Financial year")
        plt.ylabel(y_label + " (source workbook units)")
        plt.title(chart_title)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
        display(valid)

    plot_financial_series(revenue_row, "Tata Steel Revenue by Financial Year", "Revenue")
    plot_financial_series(net_income_row, "Tata Steel Net Income by Financial Year", "Net income")

## 7. Visualise historical closing price

In [ ]:
if {"Date", "Close"}.issubset(stock.columns):
    plt.figure(figsize=(12, 5))
    plt.plot(stock["Date"], stock["Close"], label="Closing Price")
    plt.xlabel("Date")
    plt.ylabel("Price")
    plt.title("Tata Steel Historical Closing Price")
    plt.legend()
    plt.tight_layout()
    plt.show()


## 8. Visualise trading volume

In [ ]:
if {"Date", "Volume"}.issubset(stock.columns):
    plt.figure(figsize=(12, 5))
    plt.plot(stock["Date"], stock["Volume"])
    plt.xlabel("Date")
    plt.ylabel("Volume")
    plt.title("Tata Steel Trading Volume")
    plt.tight_layout()
    plt.show()


## 9. Closing price with moving averages

In [ ]:
if {"Date", "Close", "MA_20", "MA_50"}.issubset(stock.columns):
    plt.figure(figsize=(12, 5))
    plt.plot(stock["Date"], stock["Close"], label="Close", alpha=0.65)
    plt.plot(stock["Date"], stock["MA_20"], label="20-day moving average")
    plt.plot(stock["Date"], stock["MA_50"], label="50-day moving average")
    plt.xlabel("Date")
    plt.ylabel("Price")
    plt.title("Closing Price and Moving Averages")
    plt.legend()
    plt.tight_layout()
    plt.show()


## 10. Distribution of daily returns

In [ ]:
if "Daily_Return" in stock.columns:
    returns = stock["Daily_Return"].dropna()
    plt.figure(figsize=(9, 5))
    sns.histplot(returns, bins=40, kde=True)
    plt.xlabel("Daily return (%)")
    plt.title("Distribution of Tata Steel Daily Returns")
    plt.tight_layout()
    plt.show()


## 11. Correlation analysis

In [ ]:
numeric_stock = stock.select_dtypes(include=np.number)
if numeric_stock.shape[1] >= 2:
    corr = numeric_stock.corr()
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Correlation Heatmap – Stock Data")
    plt.tight_layout()
    plt.show()
    display(corr)


## 12. Box plot of closing prices

In [ ]:
if "Close" in stock.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=stock["Close"])
    plt.xlabel("Closing price")
    plt.title("Distribution of Tata Steel Closing Prices")
    plt.tight_layout()
    plt.show()


## 13. Key observations

Use the outputs and charts above to write the observations for your submission:
- Overall movement in Tata Steel's closing price.
- Periods of higher and lower trading volume.
- How the moving averages compare with the closing price.
- Shape and spread of daily returns.
- Relationships between numeric stock variables.
- Revenue and net income trends, if matching rows are available in the income statement.\n- Important changes visible in the balance sheet and cash flow statement.

**Note:** Financial statement units and year labels should be taken from the source workbook. Do not assume units if they are not specified.
